# Offline CPU recommendation models

Run Surprise SVD, Cornac BPR and TF-IDF on seeded synthetic interactions and short texts. This checks installation, fitting, prediction and finite outputs. These examples do not measure production recommendation quality. No data downloads or GPU are required.

In [ ]:
import numpy as np
import pandas as pd
from recommenders.evaluation.python_evaluation import rmse
rng = np.random.default_rng(42)
rows = [(user, int(item), float(1 + (user + item) % 5))
        for user in range(30) for item in rng.choice(40, 12, replace=False)]
data = pd.DataFrame(rows, columns=['userID', 'itemID', 'rating'])
train = data.groupby('userID', group_keys=False).head(10).copy()
test = data.drop(train.index).copy()
assert len(train) == 300 and len(test) == 60

from recommenders.utils.notebook_utils import store_metadata


## Rating prediction: Surprise SVD

In [ ]:
from surprise import Dataset, Reader, SVD
from recommenders.models.surprise.surprise_utils import predict
trainset = Dataset.load_from_df(train, Reader(rating_scale=(1, 5))).build_full_trainset()
svd = SVD(n_factors=8, n_epochs=5, random_state=42)
svd.fit(trainset)
predictions = predict(svd, test)
assert len(predictions) == len(test)
assert np.isfinite(predictions['prediction']).all()
svd_rmse = rmse(test, predictions)
assert np.isfinite(svd_rmse)
print({'SVD_RMSE_synthetic': svd_rmse, 'predictions': len(predictions)})

store_metadata("svd_rmse_synthetic", svd_rmse)


## Implicit ranking: Cornac BPR

In [ ]:
import cornac
from recommenders.models.cornac.cornac_utils import predict_ranking
cornac_data = cornac.data.Dataset.from_uir(train.itertuples(index=False, name=None), seed=42)
bpr = cornac.models.BPR(k=8, max_iter=5, seed=42, num_threads=2, verbose=False)
bpr.fit(cornac_data)
ranked = predict_ranking(bpr, train, remove_seen=True)
assert len(ranked) > 0 and np.isfinite(ranked['prediction']).all()
seen = train[['userID', 'itemID']]
assert ranked.merge(seen, on=['userID', 'itemID']).empty
print({'BPR_unseen_scores': len(ranked)})

store_metadata("bpr_unseen_scores", len(ranked))


## Content similarity: TF-IDF

In [ ]:
from recommenders.models.tfidf.tfidf_utils import TfidfRecommender
texts = pd.DataFrame({'id': range(6), 'text': [
    'ocean water marine fish', 'marine ocean coral reef',
    'forest tree green leaves', 'green tree forest woodland',
    'music piano guitar sound', 'piano music rhythm melody']})
tfidf = TfidfRecommender(id_col='id', tokenization_method='none')
clean = tfidf.clean_dataframe(texts, ['text'])
vectorizer, tokens = tfidf.tokenize_text(clean)
tfidf.fit(vectorizer, tokens)
related = tfidf.recommend_top_k_items(clean, k=2)
assert len(related) == 12 and np.isfinite(related['rec_score']).all()
assert (related['id'] != related['rec_id']).all()
assert related.query('id == 0').iloc[0]['rec_id'] == 1
related

store_metadata("tfidf_recommendations", len(related))
